Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_trino_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()


In [ ]:
import pandas as pd

query = """

WITH
date_range AS (
    SELECT
        CASE
            WHEN day_of_week(current_date) = 1
                THEN current_date - interval '2' DAY
            ELSE current_date - interval '1' DAY
        END AS start_dt,
        current_date - interval '1' DAY AS end_dt
),
prompts AS (
    SELECT ticket_id, prompt, created_at FROM corp_analytics.krisha_chat_ct_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.trisha_ct_chat_tbl
    UNION
    SELECT ticket_id, prompt, created_at FROM corp_analytics.esha_ct_tbl
),
prompts1 AS (
    SELECT
        ticket_id,
        max_by(split_part(split_part(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS last_prompt,
        min_by(split_part(split_part(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS first_prompt
    FROM prompts
    WHERE split_part(split_part(prompt, 'Customer query categories List: [', 2), ']', 1) IS NOT NULL
    GROUP BY 1
)
SELECT DISTINCT
    'OUTBOUND' AS call_type,
    DATE(SUBSTRING(a.ticket_created_time, 1, 10)) AS req_date,
    a.canonical_ticket_id,
    a.csat,
    a.ticket_group AS title,
    a.first_obcs_l1_agent_name AS email_id,
    a.phone_number AS pl_before_the_ticket,
    CASE
        WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' THEN a.l1_agent_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' THEN a.l2_agent_disposition
    END AS agent_filled_reason,
    CASE
        WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_sub_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' THEN a.l1_agent_sub_disposition
        WHEN a.overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' THEN a.l2_agent_sub_disposition
    END AS agent_filled_sub_reason,
    dr.start_dt AS extracted_date1,
    dr.end_dt AS extracted_date2,
    CAST(dur.aht_seconds AS DOUBLE) / 60.0 AS AHT_in_minutes,
    dur.aht_seconds AS AHT_in_sec,
    a.latest_qrfc AS qrfc
FROM corp_analytics.cx_sub_ticket_data_model_tbl a
CROSS JOIN date_range dr
CROSS JOIN LATERAL (
    SELECT date_diff(
        'second',
        CAST(a.first_agent_assigned_time_obcs AS timestamp),
        CAST(a.resolved_time AS timestamp)
    ) AS aht_seconds
) dur
LEFT JOIN crm_tickets.tickets xz ON a.ticket_id = xz.id
LEFT JOIN prompts1 p1 ON a.ticket_id = p1.ticket_id
WHERE
    DATE(SUBSTRING(a.ticket_created_time, 1, 10)) BETWEEN dr.start_dt AND dr.end_dt
    AND a.source_channel = 'CHAT'
    AND a.overall_last_agent_type_for_sub_ticket = 'OUTBOUND_CALL_SUPPORT_AGENT_L1'
    AND (a.bot_description NOT IN (
        'Inbound_Call_Mercury_GI',
        'Inbound_Call_Mercury_HL',
        'Inbound_Call_Mercury_PL',
        'Inbound_Call_on_Mercury'
    ) OR a.bot_description IS NULL)
    AND (CASE WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_sub_disposition END)
        NOT IN (
            'Blank Call',
            'Call disconnected by customer while assisting the customer',
            'RNR',
            'Short Call',
            'Not Reachable',
            'Switch Off',
            'Not Reachable/Switch Off'
        )
    AND (CASE WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_sub_disposition END) IS NOT NULL
    AND (CASE WHEN a.overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' THEN a.obcs_l1_agent_sub_disposition END) <> ''
"""

df = get_trino_df(query)
if not df.empty:
    display(df)


In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1sCWwW9rhcxQKpvbL2u7EEP0JuF8WZ38e05PE7UJo3Eo/edit?gid=1102365521#gid=1102365521"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)